# Uncertainty with decreasing spatial support

Standard LOBO can leave close neighbours of the test borehole in training. This notebook asks a harder question: **does uncertainty react when nearby boreholes are progressively removed?**

The exclusion radius increases from 0 to 250 m while the MiniLM PCA16, XYZ, and depth configuration remains fixed.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import bovino_results as br

## 1. Buffered leave-one-borehole-out

For each held-out borehole, the training set excludes every other borehole inside the selected radius. Larger buffers reduce local support and approach a region where the model has little direct evidence.

This differs from random subsampling. The removed observations are precisely those that are most spatially relevant to the query.

In [ ]:
buffer_metrics = br.load_buffer_metrics()
display(buffer_metrics[['method', 'buffer_m', 'accuracy', 'mean_confidence',
                        'mean_total_uncertainty', 'mean_epistemic_uncertainty',
                        'error_auroc_total_uncertainty']].head())

## 2. Performance collapses as local evidence disappears

At 0 m, accuracy ranges from about 0.77 to 0.79. At 250 m, it falls to roughly 0.31 to 0.36 for every method. The common decline shows that the model relies strongly on local spatial support.

In [ ]:
fig, axes = br.plot_buffer_method_comparison()
plt.show()

## 3. Increasing uncertainty is not enough

LLLA shows the clearest MI response, rising from approximately 0.16 to 0.54. The Training Subsample Ensemble also reacts strongly. Deep Ensemble, MC Dropout, and TabICLv2 show much smaller MI changes.

Yet predictive-entropy Error AUROC drops from about 0.77 to nearly 0.50 for all methods. At the largest buffer, the models may become more uncertain on average while losing the ability to distinguish which individual predictions are wrong.

In [ ]:
methods = ['deep_ensemble', 'llla', 'mc_dropout', 'subsample_ensemble', 'tabicl_hidden']
selected = buffer_metrics[buffer_metrics['method'].isin(methods)]
start = selected[selected['buffer_m'] == 0].set_index('method')
end = selected[selected['buffer_m'] == 250].set_index('method')
columns = ['accuracy', 'mean_confidence', 'mean_total_uncertainty',
           'mean_epistemic_uncertainty', 'error_auroc_total_uncertainty']
deltas = end[columns] - start[columns]
display(deltas.round(3))

## Take-home message

Buffered LOBO supplies the clearest stress test in the study. It reveals a large dependence on nearby boreholes and separates two desirable properties: uncertainty should rise under loss of support, and it should still rank individual errors. None of the tested methods satisfies both properties consistently at large exclusion distances.